# Lab 8 — Classification with Support Vector Machines

**Course:** Data Mining Lab  
**Notebook:** Lab 8 — Classification with Support Vector Machines

## Objectives
- Compare linear and RBF SVM kernels.
- Tune C/kernel using cross-validation and report accuracy, precision, recall and F1-score.

> Run cells from top to bottom. The notebook uses a reproducible random seed. For Adult Income labs, it attempts to load the public UCI Adult dataset and uses a clearly labelled local synthetic fallback if the download is unavailable.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
COLUMNS = ["age","workclass","fnlwgt","education","education.num","marital.status",
           "occupation","relationship","race","sex","capital.gain","capital.loss",
           "hours.per.week","native.country","income"]

def load_adult():
    local = Path("adult.csv")
    if local.exists():
        data = pd.read_csv(local)
        if set(COLUMNS).issubset(data.columns):
            return data[COLUMNS].copy(), "local adult.csv"
    try:
        data = pd.read_csv(DATA_URL, names=COLUMNS, na_values="?", skipinitialspace=True)
        return data, "UCI Adult dataset (downloaded)"
    except Exception as exc:
        rng = np.random.default_rng(42)
        n = 1200
        age = rng.integers(18, 75, n)
        hours = np.clip(rng.normal(40, 12, n).round(), 1, 80).astype(int)
        edu = rng.integers(1, 16, n)
        gain = rng.choice([0,0,0,0,500,1500,5000,10000], n)
        score = -5 + 0.055*age + 0.10*edu + 0.025*hours + (gain>0)*1.0
        income = np.where(rng.random(n) < 1/(1+np.exp(-score)), ">50K", "<=50K")
        data = pd.DataFrame({
            "age":age,"workclass":rng.choice(["Private","Self-emp","Government"],n),
            "fnlwgt":rng.integers(20000,500000,n),
            "education":rng.choice(["HS-grad","Bachelors","Masters","Some-college"],n),
            "education.num":edu,"marital.status":rng.choice(["Married","Never-married","Divorced"],n),
            "occupation":rng.choice(["Tech","Sales","Service","Admin"],n),
            "relationship":rng.choice(["Husband","Not-in-family","Own-child"],n),
            "race":rng.choice(["White","Black","Other"],n),"sex":rng.choice(["Male","Female"],n),
            "capital.gain":gain,"capital.loss":rng.choice([0,0,0,500,1000],n),
            "hours.per.week":hours,"native.country":rng.choice(["United-States","Other"],n),
            "income":income})
        print("NOTICE: Public dataset unavailable; using reproducible synthetic teaching data. Reason:", str(exc)[:180])
        return data, "synthetic fallback (not UCI observations)"

df, DATA_SOURCE = load_adult()
print("Data source:", DATA_SOURCE, "| Shape:", df.shape)
display(df.head())

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, accuracy_score
from sklearn.tree import DecisionTreeClassifier

data = df.copy().replace("?", np.nan).drop_duplicates()
y = (data["income"].astype(str).str.strip().str.replace(".", "", regex=False) == ">50K").astype(int)
X = data.drop(columns=["income"])
num = X.select_dtypes(include=np.number).columns.tolist()
cat = X.select_dtypes(exclude=np.number).columns.tolist()
preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape, "Positive rate:", round(y.mean(),3))

## Linear and RBF kernel SVM

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
for kernel in ["linear","rbf"]:
    svm = Pipeline([("prep", preprocess), ("model", SVC(kernel=kernel, C=1.0, gamma="scale"))])
    svm.fit(X_train,y_train)
    pred = svm.predict(X_test)
    print("\\nKernel:",kernel,"accuracy:",round(accuracy_score(y_test,pred),4))
    print(classification_report(y_test,pred,zero_division=0))

## Hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV
svm_pipe = Pipeline([("prep", preprocess), ("model", SVC())])
grid = GridSearchCV(svm_pipe, {"model__kernel":["linear","rbf"], "model__C":[0.5,1,2]},
                    cv=3, scoring="f1", n_jobs=-1)
grid.fit(X_train,y_train)
print("Best parameters:", grid.best_params_)
print("Best cross-validation F1:", round(grid.best_score_,4))
best_pred = grid.predict(X_test)
print(classification_report(y_test,best_pred,zero_division=0))

Precision measures correctness among predicted positives; recall measures how many actual positives are found; F1 is their harmonic mean. The best model depends on the application, not accuracy alone.